<a href="https://colab.research.google.com/github/labonysur-cloud/Dashboard-/blob/main/Linkieee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes
!pip install -q trl

In [ ]:
import pandas as pd
from datasets import Dataset

file_path = 'just for linkedin - just for linkedin (1).csv'
df = pd.read_csv(file_path)

def create_prompt(row):
    return f"""### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: {row['Topic/Industry']}
- Tone: {row['Tone']}
- Goal: {row['Goal/Objective']}

### Response:
{row['Generated Content']}
"""

df['text'] = df.apply(create_prompt, axis=1)
final_dataset = Dataset.from_pandas(df[['text']])

print("Dataset successfully loaded and formatted.")
print(f"Total entries: {len(final_dataset)}")
print("\n--- Example of a formatted data entry ---")
print(final_dataset[0]['text'])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import notebook_login

# Using a publicly available model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# notebook_login() # Not strictly necessary for a public model, but good practice

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments

# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# Set up the Training Arguments
training_args = TrainingArguments(
    output_dir="linkedin_content_generator",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=20,
    learning_rate=2e-4,
)

# --- Create the Trainer (without the extra argument) ---
# This version is compatible with your library.
trainer = SFTTrainer(
    model=model,
    train_dataset=final_dataset,
    args=training_args,
    peft_config=lora_config,
)

# --- START TRAINING! ---
print("Starting model training with TinyLlama...")
trainer.train()
print("Training complete!")

# --- Save Your Fine-Tuned Model ---
trainer.save_model("linkedin_content_generator_final")

In [ ]:
from transformers import pipeline

# Load your fine-tuned model for inference
pipe = pipeline(
    "text-generation",
    model="linkedin_content_generator_final", # The path to your saved model
    tokenizer=tokenizer,
    max_new_tokens=256 # Control how long the generated text can be
)

# --- Create a new prompt to test the model ---
# Use the same structure as your training data, but leave the 'Response' part empty.
# You can change the Topic, Tone, and Goal to anything you want!

test_prompt = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: Teamwork
- Tone: Celebratory
- Goal: Build Team Morale

### Response:
"""

# --- Generate the content! ---
result = pipe(test_prompt)

# --- Print the result ---
print("\n--- Generated LinkedIn Post ---")
# The result includes the prompt, so we print just the generated part.
full_text = result[0]['generated_text']
generated_part = full_text.split("### Response:")[1]
print(generated_part.strip())

# --- Example 2: Another prompt in Bengali ---

test_prompt_bangla = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: ছাত্র জীবন (Student Life)
- Tone: চিন্তাশীল (Reflective)
- Goal: Build Connection

### Response:
"""

result_bangla = pipe(test_prompt_bangla)
full_text_bangla = result_bangla[0]['generated_text']
generated_part_bangla = full_text_bangla.split("### Response:")[1]

print("\n--- বাংলায় তৈরি লিঙ্কডইন পোস্ট ---")
print(generated_part_bangla.strip())

In [ ]:
from transformers import pipeline

language_needed = "bengali" # You can change this to "english"

# A prompt for the Bengali model
bengali_prompt = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: বই মেলা (Book Fair)
- Tone: আনন্দিত (Joyful)
- Goal: Share Experience

### Response:
"""

# A prompt for the English model
english_prompt = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: Project Management
- Tone: Professional
- Goal: Share a tip

### Response:
"""

# --- Load and use the correct model based on your need ---

if language_needed == "bengali":
    print("Loading the Bengali expert model...")
    # Load the model you trained on Bengali data
    bengali_pipe = pipeline("text-generation", model="bengali_content_generator", tokenizer=bengali_tokenizer)
    result = bengali_pipe(bengali_prompt)
else:
    print("Loading the English expert model...")
    # Load the model you trained on English data
    english_pipe = pipeline("text-generation", model="english_content_generator", tokenizer=english_tokenizer)
    result = english_pipe(english_prompt)


# Print the final result
print(result[0]['generated_text'])

In [ ]:
import pandas as pd
from datasets import Dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig
from trl import SFTTrainer

# --- Step 1: Prepare Your Bengali-Only Dataset ---

file_path = 'just for linkedin - just for linkedin (1).csv'
df = pd.read_csv(file_path)

# A simple function to detect the language
def is_bengali(text):
    if not isinstance(text, str):
        return False
    # Check if the text contains any character in the Bengali Unicode range
    return any('\u0980' <= char <= '\u09FF' for char in text)

# Filter the DataFrame to keep only the Bengali content
df_bengali = df[df['Generated Content'].apply(is_bengali)].copy()

def create_bengali_prompt(row):
    return f"""### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: {row['Topic/Industry']}
- Tone: {row['Tone']}
- Goal: {row['Goal/Objective']}

### Response:
{row['Generated Content']}
"""

df_bengali['text'] = df_bengali.apply(create_bengali_prompt, axis=1)
bengali_dataset = Dataset.from_pandas(df_bengali[['text']])

print(f"--- Successfully created Bengali-only dataset with {len(bengali_dataset)} entries ---")


# --- Step 2: Load the Multilingual Base Model and Tokenizer ---

model_id = "ai-forever/mGPT" # This model has strong Bengali support

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# This will create the 'bengali_tokenizer' that was missing
bengali_tokenizer = AutoTokenizer.from_pretrained(model_id)
bengali_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})


# --- Step 3: Train the Bengali Expert Model ---

lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir="bengali_content_generator",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=10,
    learning_rate=2e-4,
)

trainer = SFTTrainer(
    model=bengali_model,
    train_dataset=bengali_dataset,
    args=training_args,
    peft_config=lora_config,
    dataset_text_field="text",
)

print("\nStarting Bengali model training...")
trainer.train()
print("Training complete!")

trainer.save_model("bengali_content_generator_final")


# --- Step 4: Generate Content with Your Trained Bengali Model ---

print("\nLoading fine-tuned Bengali model for generation...")

# This now works because you have trained the model and have the tokenizer
bengali_pipe = pipeline(
    "text-generation",
    model="bengali_content_generator_final",
    tokenizer=bengali_tokenizer,
    max_new_tokens=256
)

bengali_prompt = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: বই মেলা (Book Fair)
- Tone: আনন্দিত (Joyful)
- Goal: Share Experience

### Response:
"""

result = bengali_pipe(bengali_prompt)

print("\n--- নতুন মডেল দিয়ে তৈরি লিঙ্কডইন পোস্ট ---")
full_text = result[0]['generated_text']
generated_part = full_text.split("### Response:")[1]
print(generated_part.strip())

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments

# --- Configure LoRA for efficient fine-tuning ---
lora_config = LoraConfig(
    r=8,
    target_modules=["c_attn"], # Note: The target modules can differ between models. 'c_attn' is common for GPT-style models.
    task_type="CAUSAL_LM",
)

# --- Set up the Training Arguments ---
training_args = TrainingArguments(
    output_dir="bengali_content_generator",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=10,
    learning_rate=2e-4,
)

# --- Create the Trainer (without the extra argument) ---
# This version is compatible with your library.
trainer = SFTTrainer(
    model=bengali_model,
    train_dataset=bengali_dataset,
    args=training_args,
    peft_config=lora_config,
)

# --- START TRAINING! ---
print("\nStarting Bengali model training...")
trainer.train()
print("Training complete!")

# --- Save Your Fine-Tuned Model ---
trainer.save_model("bengali_content_generator_final")

In [ ]:
from transformers import pipeline

print("\nLoading fine-tuned Bengali model for generation...")

# This now works because you have trained the model and have the tokenizer
bengali_pipe = pipeline(
    "text-generation",
    model="bengali_content_generator_final",
    tokenizer=bengali_tokenizer,
    max_new_tokens=256
)

# You can change the Topic, Tone, and Goal to test your model
bengali_prompt = """### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: বই মেলা (Book Fair)
- Tone: আনন্দিত (Joyful)
- Goal: Share Experience

### Response:
"""

result = bengali_pipe(bengali_prompt)

# --- Print the final, clean output ---
print("\n--- নতুন মডেল দিয়ে তৈরি লিঙ্কডইন পোস্ট ---")
full_text = result[0]['generated_text']
generated_part = full_text.split("### Response:")[1]
print(generated_part.strip())

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
from transformers import pipeline

# Load your fine-tuned model and tokenizer
pipe = pipeline(
    "text-generation",
    model="linkedin_content_generator_final", # Or your bengali model
    tokenizer=tokenizer,
    max_new_tokens=256
)

def generate_post(topic, tone, goal):
    # Format the input into the prompt structure the model expects
    prompt = f"""### Instruction:
Generate a LinkedIn post with the following properties:
- Topic: {topic}
- Tone: {tone}
- Goal: {goal}

### Response:
"""
    # Generate the text
    result = pipe(prompt)

    # Extract just the response part
    full_text = result[0]['generated_text']
    generated_part = full_text.split("### Response:")[1]
    return generated_part.strip()

# Create the Gradio interface
iface = gr.Interface(
    fn=generate_post,
    inputs=[
        gr.Textbox(lines=1, label="Topic/Industry"),
        gr.Textbox(lines=1, label="Tone"),
        gr.Textbox(lines=1, label="Goal/Objective")
    ],
    outputs=gr.Textbox(lines=5, label="Generated LinkedIn Post"),
    title="AI LinkedIn Post Generator",
    description="This AI was fine-tuned to generate professional LinkedIn content. Enter a topic, tone, and goal to see it in action."
)

# Launch the app!
iface.launch(share=True) # share=True creates a public link